In [7]:
import re

def remove_comments(sql_file_path):
    with open(sql_file_path, 'r') as file:
        sql_content = file.read()

    # Remove single-line comments starting with --
    sql_content = re.sub(r'--.*\n', '', sql_content)

    # Remove multi-line comments enclosed within /* and */
    sql_content = re.sub(r'/\*.*?\*/', '', sql_content, flags=re.DOTALL)

    # Write the modified SQL content back to the file
    with open(sql_file_path, 'w') as file:
        file.write(sql_content)

remove_comments('mysql/flughafen_mysql_all_eng.sql')

In [15]:
!docker-compose -f docker_compose/docker-compose.yml -p complex_sql up -d
#-p complex_sql benennt den container in "complex_sql"

 Container complex_sql-db-1  Starting
 Container complex_sql-db-1  Started


In [11]:
!docker exec -i docker_compose-db-1 mysql -u user -ppassword < mysql/flughafen_mysql_all_eng.sql

mysql: [Warning] Using a password on the command line interface can be insecure.


In [16]:
!docker ps

CONTAINER ID   IMAGE     COMMAND                  CREATED              STATUS         PORTS                               NAMES
168f35905da9   mysql     "docker-entrypoint.sâ€¦"   About a minute ago   Up 4 seconds   0.0.0.0:3306->3306/tcp, 33060/tcp   complex_sql-db-1


In [18]:
!docker volume ls

DRIVER    VOLUME NAME
local     557f0550ccd892e422abd2cff155def691f7eec1c035a3d78c9e70a6a8a492db
local     cc2a57d61b13d543a33cc2089822a6c8c882ed0ea07d73bf7549ca5328afb765
local     complex_sql_my-db
local     docker_compose_my-db
local     e29c189863e9cbac0c8fa6993137385e53633549c6e8e0d3cc8cae7d28eef7c5
local     mysql-docker_my-db
local     mysql_dbeaver_mysql_volume
local     myvolume


In [19]:
!docker network ls

NETWORK ID     NAME                     DRIVER    SCOPE
9599f57f71b4   bridge                   bridge    local
e844b958467b   complex_sql_default      bridge    local
c28c5984164e   docker_compose_default   bridge    local
49c203e7a00f   host                     host      local
edc7267bf01f   mysql-docker_default     bridge    local
b83a69261820   mysql_dbeaver_internal   bridge    local
2781251d84ad   mysqlnetwork             bridge    local
f9b67f9e67b2   none                     null      local


In [20]:
!docker-compose -f docker_compose/docker-compose.yml -p complex_sql down

 Container complex_sql-db-1  Stopping
 Container complex_sql-db-1  Stopped
 Container complex_sql-db-1  Removing
 Container complex_sql-db-1  Removed
 Network complex_sql_default  Removing
 Network complex_sql_default  Removed


In [21]:
!docker ps
!echo ----------------------
!docker volume ls
!echo ----------------------
!docker network ls

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES
----------------------
DRIVER    VOLUME NAME
local     557f0550ccd892e422abd2cff155def691f7eec1c035a3d78c9e70a6a8a492db
local     cc2a57d61b13d543a33cc2089822a6c8c882ed0ea07d73bf7549ca5328afb765
local     complex_sql_my-db
local     docker_compose_my-db
local     e29c189863e9cbac0c8fa6993137385e53633549c6e8e0d3cc8cae7d28eef7c5
local     mysql-docker_my-db
local     mysql_dbeaver_mysql_volume
local     myvolume
----------------------
NETWORK ID     NAME                     DRIVER    SCOPE
9599f57f71b4   bridge                   bridge    local
c28c5984164e   docker_compose_default   bridge    local
49c203e7a00f   host                     host      local
edc7267bf01f   mysql-docker_default     bridge    local
b83a69261820   mysql_dbeaver_internal   bridge    local
2781251d84ad   mysqlnetwork             bridge    local
f9b67f9e67b2   none                     null      local


In [23]:
import yaml

In [37]:
yaml_doc = """
version: '3.3'

services:
  db:
    image: mysql
    restart: always
    command: --lower_case_table_names=1

    environment:
      MYSQL_DATABASE: 'airportbase'
      MYSQL_USER: 'user'
      MYSQL_PASSWORD: 'password'
      MYSQL_ROOT_PASSWORD: 'password'

    configs:
      - source: db_init_file1
        target: /docker-entrypoint-init.d/init.sql

    ports:
      - '3306:3306'

    expose:
      - '3306'

    volumes:
      - my-db:/var/lib/mysql

volumes:
  my-db:

"""

In [38]:
print(yaml.dump(yaml_doc))

"\nversion: '3.3'\n\nservices:\n  db:\n    image: mysql\n    restart: always\n   \
  \ command: --lower_case_table_names=1\n\n    environment:\n      MYSQL_DATABASE:\
  \ 'airportbase'\n      MYSQL_USER: 'user'\n      MYSQL_PASSWORD: 'password'\n  \
  \    MYSQL_ROOT_PASSWORD: 'password'\n\n    configs:\n      - source: db_init_file1\n\
  \        target: /docker-entrypoint-init.d/init.sql\n\n    ports:\n      - '3306:3306'\n\
  \n    expose:\n      - '3306'\n\n    volumes:\n      - my-db:/var/lib/mysql\n\n\
  volumes:\n  my-db:\n\n"



In [39]:
print(yaml.safe_load(yaml_doc))

{'version': '3.3', 'services': {'db': {'image': 'mysql', 'restart': 'always', 'command': '--lower_case_table_names=1', 'environment': {'MYSQL_DATABASE': 'airportbase', 'MYSQL_USER': 'user', 'MYSQL_PASSWORD': 'password', 'MYSQL_ROOT_PASSWORD': 'password'}, 'configs': [{'source': 'db_init_file1', 'target': '/docker-entrypoint-init.d/init.sql'}], 'ports': ['3306:3306'], 'expose': ['3306'], 'volumes': ['my-db:/var/lib/mysql']}}, 'volumes': {'my-db': None}}


In [40]:
print(yaml.dump(yaml.safe_load(yaml_doc), sort_keys=False))

version: '3.3'
services:
  db:
    image: mysql
    restart: always
    command: --lower_case_table_names=1
    environment:
      MYSQL_DATABASE: airportbase
      MYSQL_USER: user
      MYSQL_PASSWORD: password
      MYSQL_ROOT_PASSWORD: password
    configs:
    - source: db_init_file1
      target: /docker-entrypoint-init.d/init.sql
    ports:
    - 3306:3306
    expose:
    - '3306'
    volumes:
    - my-db:/var/lib/mysql
volumes:
  my-db: null

